# 01 — Market Exploration

Connect to both APIs and explore what's available on each platform.

**Run this first** to verify your setup and to get a feel for the universe.


In [ ]:
import asyncio, sys, os
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from clients.kalshi_client import KalshiClient
from clients.polymarket_client import PolymarketClient
from collections import Counter
import pandas as pd

# Common Kalshi series — covers the categories that overlap with Polymarket
KALSHI_SERIES = [
    "KXPRES", "KXSEN", "KXHOUSE", "KXIMPEACH",
    "KXBTCD", "KXETHD", "KXBTC", "KXETH", "KXSOL",
    "KXFEDDECISION", "KXCPI", "KXJOBS", "KXGDP", "KXRATE",
    "KXSPX", "KXNASDAQ",
    "KXWORLDCUP", "KXNBA", "KXNFL", "KXMLB",
    "KXWAR", "KXPUTIN",
]

k = KalshiClient(environment="production")
p = PolymarketClient()
print(f"Kalshi authed={k._authed}  Polymarket authed={p._authed}  (read-only is fine for exploration)")


## Pull markets from both platforms


In [ ]:
k_markets, p_markets = await asyncio.gather(
    k.get_markets_by_series(KALSHI_SERIES, limit_per_series=50),
    p.get_markets(limit=500),
)
print(f"Kalshi: {len(k_markets)} markets")
print(f"Poly:   {len(p_markets)} markets")


## Category breakdown


In [ ]:
k_cats = Counter(m.category for m in k_markets)
p_cats = Counter(m.category for m in p_markets)
df_cats = pd.DataFrame({"kalshi": k_cats, "polymarket": p_cats}).fillna(0).astype(int)
df_cats["overlap_potential"] = df_cats[["kalshi","polymarket"]].min(axis=1)
df_cats.sort_values("overlap_potential", ascending=False)


## Sample markets side by side per category


In [ ]:
for cat in ["politics", "crypto", "economics", "sports", "geopolitics", "finance"]:
    ks = [m for m in k_markets if m.category == cat][:3]
    ps = [m for m in p_markets if m.category == cat][:3]
    if not (ks or ps): continue
    print(f"\n══ {cat.upper()} ══")
    print("  KALSHI:")
    for m in ks: print(f"    [{m.market_id[:35]:35s}] {m.title[:75]}")
    print("  POLYMARKET:")
    for m in ps: print(f"    [{m.market_id[:35]:35s}] {m.title[:75]}")


## Volume distribution (Polymarket)


In [ ]:
import plotly.express as px
df_p = pd.DataFrame([{"title": m.title[:60], "category": m.category,
                       "volume_usd": m.volume_usd} for m in p_markets])
df_p = df_p.sort_values("volume_usd", ascending=False).head(30)
fig = px.bar(df_p, y="title", x="volume_usd", color="category",
             title="Top 30 Polymarket markets by 24h volume",
             orientation="h", height=700)
fig.update_yaxes(autorange="reversed")
fig.show()


In [ ]:
await k.close(); await p.close()
print("Done.")
